[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb1v2_joint.ipynb)

# nb1v2: train the joint model (fast v2)
Fast config **v2** for `joint_v2`: T=32 frames, 32x32, base 32 channels, batch 64, lr 2e-4, **10k steps max**, sensor loss weight 10 (recorded in `config.json`), checkpoint every 2k steps with 128 DDIM-50 samples (physics + W1 + MMD, **no FVD**); FVD only on the final samples. Streams its log and appends to `progress.log` on Drive. Safe to rerun: a finished run (`log.json`) is skipped, an interrupted run resumes, nothing on Drive is overwritten (new run name `joint_v2`). **To stop early:** interrupt the training cell, then run the *finalize* cell (newest EMA checkpoint -> final samples + `log.json`). **Start this one first.**

Needs `nb0_benchmark` finished (it decides FVD vs KID once). **If you start several v2 notebooks at once, start the joint one first** (it builds the T=32 cache; the others wait/retry).

## 1. GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> A100 GPU, then rerun.")
print("cuda:", torch.cuda.get_device_name(0), "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

## 3. Code

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

## 4. Install

In [ ]:
# torch / torchvision come with Colab. pytorchvideo is the second I3D source for FVD (see videofeat.py).
!pip install -q scipy matplotlib pytorchvideo

## 5. T=32 environment + cache

In [ ]:
# v2 runs use T=32 frames. These must be set BEFORE cvpr27 is imported (physvid reads them at import).
%env PV_T=32
%env PV_DT=0.04
%env PV_WIN=8

In [ ]:
%%bash
# T=32 cache (separate file names from the T=64 cache). Retries cover the case where another notebook is building it right now.
export PV_T=32 PV_DT=0.04 PV_WIN=8
cd /content/repo
for i in 1 2 3 4 5 6; do python -m cvpr27.build_cache && break; echo "cache busy, retry $i"; sleep 20; done
ls -la /content/drive/MyDrive/physgen_cvpr27/cache/ | grep T32
test -f /content/drive/MyDrive/physgen_cvpr27/cache/video_metric.json || (echo "MISSING video_metric.json: run nb0_benchmark first" && exit 1)

## 6. TIMING (50 training steps + one sampling batch). Read the projection before launching.

In [ ]:
%%bash
export PV_T=32 PV_DT=0.04 PV_WIN=8
cd /content/repo
PYTHONUNBUFFERED=1 python -u -m cvpr27.train --variant joint --run_name joint_v2 --preset v2 --time_only

## 7. Train `joint_v2` (streams; also appended to `joint_v2/progress.log` on Drive)

In [ ]:
%%bash
set -o pipefail
export PV_T=32 PV_DT=0.04 PV_WIN=8
mkdir -p "/content/drive/MyDrive/physgen_cvpr27/joint_v2"
cd /content/repo
PYTHONUNBUFFERED=1 python -u -m cvpr27.train --variant joint --run_name joint_v2 --preset v2 --seed 0 2>&1 | tee -a "/content/drive/MyDrive/physgen_cvpr27/joint_v2/console_$(date +%s).log"

## 8. Finalize (only if you stopped early, or the run did not reach `log.json`)
Newest `ema_*.pt` -> final 1,024 ancestral samples + metrics (with the video metric) + `log.json`. Skipped automatically if the run already finished.

In [ ]:
%%bash
set -o pipefail
export PV_T=32 PV_DT=0.04 PV_WIN=8
cd /content/repo
test -f "/content/drive/MyDrive/physgen_cvpr27/joint_v2/log.json" && echo "already finished (log.json exists)" || PYTHONUNBUFFERED=1 python -u -m cvpr27.train --variant joint --run_name joint_v2 --preset v2 --finalize 2>&1 | tee -a "/content/drive/MyDrive/physgen_cvpr27/joint_v2/console_finalize_$(date +%s).log"

## 9. Summary (read from the run's files)

In [ ]:
%%bash
export PV_T=32 PV_DT=0.04 PV_WIN=8
cd /content/repo && python -m cvpr27.summary --run joint_v2